# The Rent Is Right

A multi-agent system that scans rental listings across **New York, Lagos, and Nairobi**, estimates fair market rent using an ensemble of AI models, and alerts users to underpriced deals.

## Concepts Covered
- Modal.com serverless GPU deployment (SpecialistAgent)
- RAG with Chroma vector database (FrontierAgent)
- Weighted ensemble of multiple models
- OpenAI Structured Outputs (ScannerAgent)
- OpenAI function calling / agentic loop (AutonomousAgent)
- Push notifications via Pushover API (MessagingAgent)
- Gradio interactive dashboard with 3D visualization

## Setup

Create a `.env` file in this directory with:
```
OPENAI_API_KEY=your_key
PUSHOVER_TOKEN=your_token (optional)
PUSHOVER_USER=your_user (optional)
```

In [ ]:
!uv pip install openai chromadb sentence-transformers gradio plotly scikit-learn modal pydantic requests python-dotenv datasets

In [ ]:
import sys
sys.path.append('.')

from dotenv import load_dotenv
load_dotenv()

## Step 1: Data Models & Mock Listings

We define Pydantic models for type-safe data flow between agents, and a mock feed that generates realistic rental listings for our three target cities.

In [ ]:
from agents.rental_deals import (
    ScrapedListing, RentalDeal, DealSelection, RentalOpportunity,
    generate_mock_listings, fetch_all_listings, scraped_to_deal
)

# Generate sample listings for each city
for city in ["New York", "Lagos", "Nairobi"]:
    listings = generate_mock_listings(city, count=3)
    print(f"\n--- {city} ---")
    for l in listings:
        print(f"  {l.title} | ${l.price:,.2f}/mo")

## Step 2: ScannerAgent — Deal Discovery with Structured Outputs

The ScannerAgent fetches all listings and asks GPT-4o-mini to select the 5 best deals. It uses OpenAI Structured Outputs (`response_format=DealSelection`) so the response is a validated Pydantic object — no manual JSON parsing needed.

In [ ]:
from agents.rental_scanner_agent import RentalScannerAgent

scanner = RentalScannerAgent()
deals = scanner.scan(count_per_city=5)

for d in deals:
    print(f"{d.title} | {d.bedrooms}BR | {d.sqft}sqft | ${d.rent:,.2f}/mo")

## Step 3: Fetch Dataset & Set Up Vector Database

We fetch our curated rental dataset from HuggingFace, then embed the listings into a Chroma vector database using `all-MiniLM-L6-v2`. This is what the FrontierAgent queries for RAG context.

In [ ]:
from datasets import load_dataset
import chromadb
from sentence_transformers import SentenceTransformer

# Fetch curated rental dataset from HuggingFace
dataset = load_dataset("Gasmyr/rental-prices", split="train")
print(f"Loaded {len(dataset)} rental listings from HuggingFace.")

# Set up embedding model and Chroma
model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="rental_vectorstore")
collection = client.get_or_create_collection("rental_listings")

# Reset if vector store is out of sync with dataset
if collection.count() != len(dataset):
    print(f"Vector DB has {collection.count()} items but dataset has {len(dataset)}. Resetting...")
    client.delete_collection("rental_listings")
    collection = client.get_or_create_collection("rental_listings")

# Populate vector DB with dataset (batch encoding for speed)
if collection.count() == 0:
    texts = [
        f"{row['bedrooms']}BR {row['sqft']}sqft in {row['city']}. {row['description']}"
        for row in dataset
    ]
    ids = [f"listing_{i}" for i in range(len(dataset))]
    metadatas = [
        {"city": row["city"], "rent": float(row["rent"]), "bedrooms": row["bedrooms"]}
        for row in dataset
    ]

    BATCH_SIZE = 500
    for start in range(0, len(texts), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(texts))
        batch_texts = texts[start:end]
        batch_embeddings = model.encode(batch_texts, show_progress_bar=False).tolist()
        collection.add(
            ids=ids[start:end],
            embeddings=batch_embeddings,
            documents=batch_texts,
            metadatas=metadatas[start:end],
        )
        print(f"  Embedded {end}/{len(texts)} listings...")

    print(f"Added {collection.count()} listings to vector DB.")
else:
    print(f"Vector DB already has {collection.count()} listings.")

## Step 4: FrontierAgent — RAG-Based Rent Estimation

The FrontierAgent encodes the target property, retrieves the 5 most similar listings from Chroma, and sends them as context to GPT-4o. The model uses these comparable properties to estimate fair rent.

In [ ]:
from agents.rental_frontier_agent import RentalFrontierAgent

frontier = RentalFrontierAgent(collection=collection)

# Estimate fair rent for the first scanned deal
if deals:
    test_deal = deals[0]
    estimate = frontier.estimate(test_deal)
    print(f"\nListed: ${test_deal.rent:,.2f}/mo")
    print(f"Estimated fair rent: ${estimate:,.2f}/mo")
    print(f"Difference: ${estimate - test_deal.rent:,.2f}/mo")

## Step 4b: SpecialistAgent on Modal

The fine-tuned model weights are already published at `Gasmyr/rental-pricer` on HuggingFace. **You do not need to retrain.** You only need to deploy the inference service on your own Modal account.

**First-time setup:**
```bash
modal setup                  # one-time authentication
modal secret create huggingface-secret HF_TOKEN=your_hf_token
```

**Deploy inference service (uses the pre-trained weights):**
```bash
modal deploy rental_pricer_service.py
```

**Only if you want to retrain from scratch (~1-3 hours, ~$2-5):**
```bash
modal run rental_pricer_service.py::train
```

If Modal is not deployed, the ensemble will gracefully fallback to the Frontier agent.

In [ ]:
# Optional: test the Modal service directly before using it in the ensemble
import modal

try:
    estimate_rent = modal.Function.from_name("rental-pricer-service", "estimate_rent")
    test_result = estimate_rent.remote("2-bedroom, 850 sqft in New York. Modern kitchen, near public transit.")
    print(f"Modal specialist estimate: ${float(test_result):,.2f}/mo")
except Exception as e:
    print(f"Modal not deployed yet: {e}\nRun 'modal deploy rental_pricer_service.py' first, or skip — ensemble will fallback.")

## Step 5: EnsembleAgent — Weighted Model Combination

The EnsembleAgent combines three models:
- **80%** FrontierAgent (RAG + GPT) — most accurate
- **10%** SpecialistAgent (fine-tuned Llama on Modal)
- **10%** Neural Network

Note: SpecialistAgent requires Modal deployment. For local testing, the ensemble gracefully falls back.

In [ ]:
from agents.rental_ensemble_agent import RentalEnsembleAgent
from agents.rental_specialist_agent import RentalSpecialistAgent

# SpecialistAgent requires Modal deployment — set to None if not deployed yet
try:
    specialist = RentalSpecialistAgent()
except Exception as e:
    print(f"Modal not available ({e}). Ensemble will fallback to Frontier for specialist weight.")
    specialist = None

ensemble = RentalEnsembleAgent(
    frontier=frontier,
    specialist=specialist,
)

if deals:
    test_deal = deals[0]
    estimate = ensemble.estimate(test_deal)
    print(f"\nEnsemble estimate: ${estimate:,.2f}/mo")

## Step 6: MessagingAgent — Push Notifications

The MessagingAgent uses Claude Sonnet to craft a compelling alert, then sends it via Pushover. If Pushover credentials aren't set, it logs the message instead.

In [ ]:
from agents.rental_messaging_agent import RentalMessagingAgent
from agents.rental_deals import RentalOpportunity

messenger = RentalMessagingAgent()

if deals:
    test_opp = RentalOpportunity(
        deal=deals[0],
        estimated_fair_rent=estimate,
        monthly_savings=max(estimate - deals[0].rent, 0),
    )
    messenger.alert(test_opp)

## Step 7: AutonomousAgent — Agentic Loop with Tool Use

The AutonomousAgent uses OpenAI function calling to self-direct. It defines 3 tools (scan, estimate, alert) and GPT decides the execution order autonomously via an agentic while-loop.

In [ ]:
from agents.rental_autonomous_agent import RentalAutonomousAgent

autonomous = RentalAutonomousAgent(
    scanner=scanner,
    ensemble=ensemble,
    messenger=messenger,
)

opportunities = autonomous.run()

print(f"\n=== Found {len(opportunities)} opportunities ===")
for opp in sorted(opportunities, key=lambda o: o.monthly_savings, reverse=True):
    print(f"{opp.deal.title} | Listed: ${opp.deal.rent:,.0f} | Fair: ${opp.estimated_fair_rent:,.0f} | Savings: ${opp.monthly_savings:,.0f}")

## Step 8: 3D Vector Visualization

Visualize the Chroma vector database in 3D. Listings cluster by city, confirming the embeddings capture geographic and pricing patterns.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA

result = collection.get(include=["embeddings", "metadatas", "documents"])

if result["embeddings"] is not None and len(result["embeddings"]) > 0:
    embeddings = np.array(result["embeddings"])
    coords = PCA(n_components=3).fit_transform(embeddings)

    city_colors = {"New York": "red", "Lagos": "green", "Nairobi": "blue"}
    colors = [city_colors.get(m.get("city", ""), "gray") for m in result["metadatas"]]
    labels = [d[:60] for d in result["documents"]]

    fig = go.Figure(data=[go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode="markers",
        marker=dict(size=3, color=colors, opacity=0.7),
        text=labels, hoverinfo="text",
    )])
    fig.update_layout(title="Rental Listings Vector Space", height=500)
    fig.show()
else:
    print("No embeddings in vector DB yet.")

## Step 9: Launch the Gradio Dashboard

Run the full interactive UI with deal hunting, alerts, and vector visualization.

In [ ]:
from the_rent_is_right import create_ui

demo = create_ui()
demo.launch()